In [1]:
import os, csv, time
import numpy as np
from datetime import datetime, timezone

import nidaqmx
from nidaqmx.constants import AcquisitionType, Edge, CountDirection, RegenerationMode
from nidaqmx.stream_readers import CounterReader
from nidaqmx.stream_writers import AnalogMultiChannelWriter

# ---------- helpers ----------

def _now_iso_utc():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%z")

def _diff_u32(cum):
    x = cum.astype(np.uint64)
    d = np.diff(x, prepend=x[0])
    wrap = (d.astype(np.int64) < 0)
    d[wrap] += 2**32
    d[0] = 0
    return d.astype(np.uint32)

def _validate_voltages_0_5(v):
    v = np.asarray(v, float)
    if v.ndim != 1 or v.size < 1:
        raise ValueError("voltage sequence must be 1D, length ≥ 1")
    vmin, vmax = float(v.min()), float(v.max())
    if vmin < 0.0 or vmax > 5.0:
        raise ValueError(f"Voltages must be within [0,5] V; found [{vmin:.4g}, {vmax:.4g}]")
    return v

def _validate_voltages_0_10(v):
    v = np.asarray(v, float)
    if v.ndim != 1 or v.size < 1:
        raise ValueError("voltage sequence must be 1D, length ≥ 1")
    vmin, vmax = float(v.min()), float(v.max())
    if vmin < 0.0 or vmax > 10.0:
        raise ValueError(f"Voltages must be within [0,10] V; found [{vmin:.4g}, {vmax:.4g}]")
    return v

def _build_names(out_dir, bin_rate_req, ao_rate, bin_rate_real, voa_rate_req, voa_rate_real, ao_mode):
    stamp = _now_iso_utc()
    base = (f"ni6366_binReq-{bin_rate_req:g}Hz_binReal-{bin_rate_real:g}Hz"
            f"_AO-{ao_rate:g}Hz_VOAreq-{voa_rate_req:g}Hz_VOAreal-{voa_rate_real:g}Hz"
            f"_mode-{ao_mode}_{stamp}")
    return (os.path.join(out_dir, base + ".npz"),
            os.path.join(out_dir, base + ".csv"),
            os.path.join(out_dir, base + "_meta.txt"))

def _ao_step_sequences_fast(ao_rate_hz, voa_rate_hz, vs_voa, vs_pm, mode, total_fast_samples=None):
    """
    Build AO buffers at the FAST rate for *both* channels, stepping together:
      Kf = round(ao_rate / voa_rate); realized step rate = ao_rate / Kf
      repeat mode  -> one-cycle buffers (len = len(seq)*Kf), regen
      finite mode  -> exact total_fast_samples (wrap sequences as needed)
    Returns: (buf_voa, buf_pm, voa_rate_real, Kf)
    """
    v_voa = _validate_voltages_0_5(vs_voa)
    v_pm  = _validate_voltages_0_10(vs_pm)
    if voa_rate_hz <= 0 or ao_rate_hz <= 0:
        raise ValueError("Rates must be positive.")
    Kf = max(1, int(round(ao_rate_hz / voa_rate_hz)))
    voa_real = ao_rate_hz / Kf

    if mode == "repeat":
        buf_voa = np.repeat(v_voa, Kf)
        buf_pm  = np.repeat(v_pm,  Kf)
    else:
        if total_fast_samples is None:
            raise ValueError("total_fast_samples required for finite AO")
        steps_needed = int(np.ceil(total_fast_samples / Kf))
        seq_voa = np.repeat(v_voa[np.arange(steps_needed) % v_voa.size], Kf)
        seq_pm  = np.repeat(v_pm [np.arange(steps_needed) % v_pm .size], Kf)
        buf_voa = seq_voa[:total_fast_samples]
        buf_pm  = seq_pm [:total_fast_samples]

    return buf_voa, buf_pm, voa_real, Kf

# ---------- main (fast clock + decimate, dual AO) ----------

def run_experiment_fastclock_dual(
    *,
    bin_rate_hz: float,              # requested bin rate (saved counts cadence)
    ao_rate_hz: float,               # FAST master clock for CI & AO (>= bin_rate_hz)
    voa_rate_hz: float,              # step rate for BOTH AO channels
    vs_voa: np.ndarray,              # VOA step values (0..5 V)
    vs_pm:  np.ndarray,              # Phase-mod step values (0..5 V)
    duration_seconds: float = 30.0,  # total run time
    ao_mode: str = "repeat",         # "repeat" (regen) or "finite"
    device: str = "Dev1",
    spad_pfi: str = "/Dev1/PFI0",    # SPAD TTL source
    ao_chan_voa: str = "/Dev1/ao0",
    ao_chan_pm:  str = "/Dev1/ao1",
    ctr_count: str = "ctr0",
    ctr_clk:   str = "ctr1",
    clk_term:  str = "/Dev1/Ctr1InternalOutput",  # or a PFI you export ctr1 to
    out_dir: str = ".",
    chunk_seconds: float = 2.0       # read chunk at FAST rate
):
    """
    AO0 (VOA) and AO1 (PM) are clocked by Ctr1 at ao_rate_hz (fast) and step together at voa_rate_hz.
    CI is also sampled at ao_rate_hz; we sum M=round(ao_rate/bin_rate) fast samples per bin.
    Returns (timestamps[datetime64[ns]] at bin cadence, counts_per_bin[uint32], voa_volts_per_bin[float], pm_volts_per_bin[float]).
    Saves NPZ + CSV + META.
    """
    os.makedirs(out_dir, exist_ok=True)
    if ao_mode not in ("repeat", "finite"):
        raise ValueError("ao_mode must be 'repeat' or 'finite'")
    if ao_rate_hz < bin_rate_hz:
        raise ValueError("ao_rate_hz must be >= bin_rate_hz")
    if bin_rate_hz <= 0 or ao_rate_hz <= 0 or voa_rate_hz <= 0:
        raise ValueError("Rates must be positive.")

    # Decimation ratio and realized bin rate
    M = max(1, int(round(ao_rate_hz / bin_rate_hz)))
    bin_rate_real = ao_rate_hz / M
    total_bins = int(np.round(duration_seconds * bin_rate_real))
    total_fast = total_bins * M

    # AO buffers (fast rate) for both channels
    if ao_mode == "repeat":
        ao_buf_voa, ao_buf_pm, voa_rate_real, Kf = _ao_step_sequences_fast(
            ao_rate_hz, voa_rate_hz, vs_voa, vs_pm, "repeat")
        ao_sample_mode = AcquisitionType.CONTINUOUS
        ao_regen = RegenerationMode.ALLOW_REGENERATION
        ao_samples_to_write = len(ao_buf_voa)
    else:
        ao_buf_voa, ao_buf_pm, voa_rate_real, Kf = _ao_step_sequences_fast(
            ao_rate_hz, voa_rate_hz, vs_voa, vs_pm, "finite", total_fast_samples=total_fast)
        ao_sample_mode = AcquisitionType.FINITE
        ao_regen = RegenerationMode.DONT_ALLOW_REGENERATION
        ao_samples_to_write = len(ao_buf_voa)

    # File names
    npz_path, csv_path, meta_path = _build_names(
        out_dir, bin_rate_hz, ao_rate_hz, bin_rate_real, voa_rate_hz, voa_rate_real, ao_mode
    )

    # Storage (bin cadence)
    counts_bins = np.empty(0, dtype=np.uint32)
    voa_bins    = np.empty(0, dtype=float)
    pm_bins     = np.empty(0, dtype=float)
    times_bins  = np.empty(0, dtype='datetime64[ns]')

    # Chunk sizing (fast rate)
    chunk_fast = max(M * 2, int(round(chunk_seconds * ao_rate_hz)))

    # Tasks
    clk_task = nidaqmx.Task()
    cnt_task = nidaqmx.Task()
    ao_task  = nidaqmx.Task()

    try:
        # Clock: Ctr1 @ ao_rate_hz, continuous
        clk_task.co_channels.add_co_pulse_chan_freq(f"/{device}/{ctr_clk}", freq=ao_rate_hz, duty_cycle=0.5)
        clk_task.timing.cfg_implicit_timing(sample_mode=AcquisitionType.CONTINUOUS)

        # CI: Ctr0 sampled by fast clock
        ch = cnt_task.ci_channels.add_ci_count_edges_chan(f"/{device}/{ctr_count}",
                                                          edge=Edge.RISING,
                                                          count_direction=CountDirection.COUNT_UP)
        ch.ci_count_edges_term = spad_pfi
        cnt_task.timing.cfg_samp_clk_timing(rate=ao_rate_hz, source=clk_term,
                                            active_edge=Edge.RISING,
                                            sample_mode=AcquisitionType.CONTINUOUS,
                                            samps_per_chan=chunk_fast)
        cnt_reader = CounterReader(cnt_task.in_stream)

        # AO: TWO channels clocked by fast clock
        ao_task.ao_channels.add_ao_voltage_chan(ao_chan_voa, min_val=0.0, max_val=5.0)  # VOA
        ao_task.ao_channels.add_ao_voltage_chan(ao_chan_pm,  min_val=0.0, max_val=10.0)  # PM
        ao_task.timing.cfg_samp_clk_timing(rate=ao_rate_hz, source=clk_term,
                                           active_edge=Edge.RISING,
                                           sample_mode=ao_sample_mode,
                                           samps_per_chan=ao_samples_to_write)
        ao_task.out_stream.regen_mode = ao_regen

        # Write both channels in one shot (shape = [num_channels, num_samples])
        ao_writer = AnalogMultiChannelWriter(ao_task.out_stream, auto_start=False)
        if ao_mode == "repeat":
            # both same length
            ao_writer.write_many_sample(np.vstack([ao_buf_voa, ao_buf_pm]))
        else:
            ao_writer.write_many_sample(np.vstack([ao_buf_voa, ao_buf_pm]))

        # CSV header
        with open(csv_path, "w", newline="") as fcsv:
            csv.writer(fcsv).writerow(["iso_end_time_utc", "count_bin", "voa_V", "pm_V"])

        # Start order: arm consumers, then clock
        cnt_task.start()
        ao_task.start()
        t0_wall = np.datetime64(datetime.now(timezone.utc))
        clk_task.start()

        remaining_fast = total_fast
        ao_cycle_len = len(ao_buf_voa)  # == len(ao_buf_pm) in repeat mode

        while remaining_fast > 0:
            n_fast_read = min(chunk_fast, remaining_fast)
            n_fast_read_eff = min(n_fast_read + 1, remaining_fast + 1)

            buf = np.zeros(n_fast_read_eff, dtype=np.uint32)
            timeout = (n_fast_read_eff / ao_rate_hz) + 2.0
            cnt_reader.read_many_sample_uint32(buf, number_of_samples_per_channel=n_fast_read_eff, timeout=timeout)
            d_fast = _diff_u32(buf)[1:]  # per-fast-sample counts

            usable = (d_fast.size // M) * M
            if usable == 0:
                continue

            d_fast = d_fast[:usable]
            nb = usable // M
            d_bins = d_fast.reshape(nb, M).sum(axis=1).astype(np.uint32)

            produced_fast = total_fast - remaining_fast
            fast_idxs = produced_fast + np.arange(usable)

            # AO values that actually played — take last fast sample in each bin for BOTH channels
            if ao_mode == "repeat":
                ao_fast_voa = ao_buf_voa[fast_idxs % ao_cycle_len]
                ao_fast_pm  = ao_buf_pm [fast_idxs % ao_cycle_len]
            else:
                ao_fast_voa = ao_buf_voa[fast_idxs]
                ao_fast_pm  = ao_buf_pm [fast_idxs]

            voa_b = ao_fast_voa.reshape(nb, M)[:, -1].astype(float)
            pm_b  = ao_fast_pm .reshape(nb, M)[:, -1].astype(float)

            # Bin end timestamps on realized bin grid
            bin_start_idx = (produced_fast // M)
            t_chunk = t0_wall + (1e9 * (np.arange(nb) + (bin_start_idx + 1)) / bin_rate_real).astype("timedelta64[ns]")

            # Append & CSV
            counts_bins = np.concatenate([counts_bins, d_bins])
            voa_bins    = np.concatenate([voa_bins,    voa_b ])
            pm_bins     = np.concatenate([pm_bins,     pm_b  ])
            times_bins  = np.concatenate([times_bins,  t_chunk])

            with open(csv_path, "a", newline="") as fcsv:
                w = csv.writer(fcsv)
                for ti, ci_bin, v_voa, v_pm in zip(t_chunk, d_bins, voa_b, pm_b):
                    w.writerow([str(ti.astype('datetime64[us]')), int(ci_bin), float(v_voa), float(v_pm)])

            remaining_fast -= usable

        # Stop tasks
        clk_task.stop(); ao_task.stop(); cnt_task.stop()

        # Save NPZ
        np.savez(
            npz_path,
            timestamps=times_bins,
            counts=counts_bins,
            voa_volts=voa_bins,
            pm_volts=pm_bins,
            t0_utc=str(t0_wall),
            bin_rate_hz_requested=float(bin_rate_hz),
            bin_rate_hz_realized=float(bin_rate_real),
            ao_rate_hz=float(ao_rate_hz),
            voa_rate_hz_requested=float(voa_rate_hz),
            voa_rate_hz_realized=float(voa_rate_real),
            M_fast_per_bin=int(M),
            Kf_steps_per_voa=int(Kf),
        )

        # Meta
        with open(meta_path, "w") as fm:
            fm.write(
                f"device={device}\n"
                f"spad_pfi={spad_pfi}\n"
                f"ao_chan_voa={ao_chan_voa}\n"
                f"ao_chan_pm={ao_chan_pm}\n"
                f"ctr_count={ctr_count}\n"
                f"ctr_clk={ctr_clk}\n"
                f"clk_term={clk_term}\n"
                f"ao_rate_hz={ao_rate_hz}\n"
                f"bin_rate_hz_requested={bin_rate_hz}\n"
                f"bin_rate_hz_realized={bin_rate_real}\n"
                f"M_fast_per_bin={M}\n"
                f"voa_rate_hz_requested={voa_rate_hz}\n"
                f"voa_rate_hz_realized={voa_rate_real}\n"
                f"Kf_steps_per_voa={Kf}\n"
                f"ao_mode={ao_mode}\n"
                f"chunk_seconds={chunk_seconds}\n"
                f"duration_seconds={duration_seconds}\n"
                f"timestamp_start_utc={datetime.now(timezone.utc).isoformat()}\n"
            )

        return times_bins, counts_bins, voa_bins, pm_bins

    finally:
        for t in (ao_task, cnt_task, clk_task):
            try: t.stop()
            except: pass
            try: t.close()
            except: pass

In [49]:
vs_pulses=[]
p_height=np.linspace(0, 10, 51, endpoint=True)
for i in range(len(p_height)):
    vs_pulses.append(p_height[i])
vs_pulses.append(0)

In [9]:
vs_pulses2=[]
for i in range(26):
    vs_pulses2.append(1)
for i in range(26):
    vs_pulses2.append(0)

In [53]:
len(vs_pulses2)

52

In [55]:
len(vs_pulses)

52

In [57]:
vs_pulses

[0.0,
 0.2,
 0.4,
 0.6000000000000001,
 0.8,
 1.0,
 1.2000000000000002,
 1.4000000000000001,
 1.6,
 1.8,
 2.0,
 2.2,
 2.4000000000000004,
 2.6,
 2.8000000000000003,
 3.0,
 3.2,
 3.4000000000000004,
 3.6,
 3.8000000000000003,
 4.0,
 4.2,
 4.4,
 4.6000000000000005,
 4.800000000000001,
 5.0,
 5.2,
 5.4,
 5.6000000000000005,
 5.800000000000001,
 6.0,
 6.2,
 6.4,
 6.6000000000000005,
 6.800000000000001,
 7.0,
 7.2,
 7.4,
 7.6000000000000005,
 7.800000000000001,
 8.0,
 8.200000000000001,
 8.4,
 8.6,
 8.8,
 9.0,
 9.200000000000001,
 9.4,
 9.600000000000001,
 9.8,
 10.0,
 0]

In [61]:
if __name__ == "__main__":
    # Example: bins ≈1 kHz; AO/CI at 10 kHz; steps at 50 Hz for BOTH channels.
    vs_voa = vs_pulses2          # VOA step values
    #vs_pm  = np.array([0.0, 1.2, 2.4, 3.6, 4.8])            # PM  step values
    vs_pm  = vs_pulses            # PM  step values
    ts, cts, v_voa, v_pm = run_experiment_fastclock_dual(
        bin_rate_hz=1000.0,
        ao_rate_hz=100000.0,
        voa_rate_hz=52000.0,
        vs_voa=vs_voa,
        vs_pm=vs_pm,
        duration_seconds=180.0,
        ao_mode="repeat",
        device="Dev1",
        spad_pfi="/Dev1/PFI8",
        ao_chan_voa="/Dev1/ao1",
        ao_chan_pm="/Dev1/ao0",
        ctr_count="ctr0",
        ctr_clk="ctr1",
        clk_term="/Dev1/Ctr1InternalOutput",
        out_dir=".",
        chunk_seconds=1.0,
    )
    print("Bins:", cts.size, "| mean cps/bin ~", cts.mean())


C:\Users\labuser\AppData\Local\Temp\ipykernel_3288\2531111776.py:193: DeprecationWarning: parsing timezone aware datetimes is deprecated; this will raise an error in the future
  t0_wall = np.datetime64(datetime.now(timezone.utc))


Bins: 180000 | mean cps/bin ~ 0.0


In [63]:
vs_pulses[34]

6.800000000000001

In [65]:
vs_pulses[35]

7.0

In [11]:
vs_pulses2=[]
for i in range(26):
    vs_pulses2.append(2.6)
for i in range(26):
    vs_pulses2.append(2.6)

In [13]:
vs_pulses3=[]
p_height=np.linspace(0, 6.86, 52, endpoint=True)
for i in range(len(p_height)):
    vs_pulses3.append(p_height[i])

In [15]:
len(vs_pulses3)

52

In [23]:
if __name__ == "__main__":
    # Example: bins ≈1 kHz; AO/CI at 10 kHz; steps at 50 Hz for BOTH channels.
    vs_voa = vs_pulses2          # VOA step values
    #vs_pm  = np.array([0.0, 1.2, 2.4, 3.6, 4.8])            # PM  step values
    vs_pm  = vs_pulses3            # PM  step values
    ts, cts, v_voa, v_pm = run_experiment_fastclock_dual(
        bin_rate_hz=1000.0,
        ao_rate_hz=100000.0,
        voa_rate_hz=5.20,
        vs_voa=vs_voa,
        vs_pm=vs_pm,
        duration_seconds=180.0,
        ao_mode="repeat",
        device="Dev1",
        spad_pfi="/Dev1/PFI8",
        ao_chan_voa="/Dev1/ao1",
        ao_chan_pm="/Dev1/ao0",
        ctr_count="ctr0",
        ctr_clk="ctr1",
        clk_term="/Dev1/Ctr1InternalOutput",
        out_dir=".",
        chunk_seconds=1.0,
    )
    print("Bins:", cts.size, "| mean cps/bin ~", cts.mean())


C:\Users\labuser\AppData\Local\Temp\ipykernel_4064\2531111776.py:193: DeprecationWarning: parsing timezone aware datetimes is deprecated; this will raise an error in the future
  t0_wall = np.datetime64(datetime.now(timezone.utc))


Bins: 180000 | mean cps/bin ~ 0.0


In [ ]:
v2pi=6.86

In [3]:
phis=np.load('day_2_long_phis.npy')
etas=np.load('day_2_long_vs.npy')

In [7]:
if __name__ == "__main__":
    # Example: bins ≈1 kHz; AO/CI at 10 kHz; steps at 50 Hz for BOTH channels.
    vs_voa = etas          # VOA step values
    #vs_pm  = np.array([0.0, 1.2, 2.4, 3.6, 4.8])            # PM  step values
    vs_pm  = phis            # PM  step values
    ts, cts, v_voa, v_pm = run_experiment_fastclock_dual(
        bin_rate_hz=1000.0,
        ao_rate_hz=100000.0,
        voa_rate_hz=2000.0,
        vs_voa=vs_voa,
        vs_pm=vs_pm,
        duration_seconds=180.0,
        ao_mode="repeat",
        device="Dev1",
        spad_pfi="/Dev1/PFI8",
        ao_chan_voa="/Dev1/ao1",
        ao_chan_pm="/Dev1/ao0",
        ctr_count="ctr0",
        ctr_clk="ctr1",
        clk_term="/Dev1/Ctr1InternalOutput",
        out_dir=".",
        chunk_seconds=1.0,
    )
    print("Bins:", cts.size, "| mean cps/bin ~", cts.mean())


C:\Users\labuser\AppData\Local\Temp\ipykernel_4064\2531111776.py:193: DeprecationWarning: parsing timezone aware datetimes is deprecated; this will raise an error in the future
  t0_wall = np.datetime64(datetime.now(timezone.utc))


Bins: 180000 | mean cps/bin ~ 0.0
